# 06 · Benchmark CPU vs GPU (para Robson — Etapa 5)

Compara **el mismo modelo, entrenado sobre los mismos datos, forzando CPU vs forzando GPU**,
para tener un speedup real y consolidado — no datos sueltos de distintas corridas.

Cubre lo que pidió Robson:
- Tiempos de entrenamiento tabular (cuML) en CPU vs GPU
- Uso de RAM y GPU durante el entrenamiento
- Tabla + gráfico de speedup, listos para su análisis de rendimiento y dashboard

**Entrada:** `../datos/train.parquet`, `val.parquet`, `test.parquet` (ya generados por `01_exploracion_dataset.ipynb`)
**Salidas:** `../resultados/tablas/05_benchmark_cpu_vs_gpu.csv`, `../resultados/figuras/08_speedup_cpu_vs_gpu.png`


In [ ]:
import os, json, time, threading, subprocess
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    import psutil
    HAY_PSUTIL = True
except ImportError:
    HAY_PSUTIL = False
    print("Aviso: instala psutil para medir RAM (pip install --user psutil)")

# Datos pesados (splits train/val/test) -> viven en /data, NO en /home
USUARIO = os.path.basename(os.path.expanduser("~"))
DATOS_BASE_DIR = os.path.join("/data", USUARIO, "deteccion_incendios")
# Resultados livianos (tablas/figuras que van al repo de Git) -> se quedan en /home
BASE_DIR = os.path.join(os.path.expanduser("~"), "deteccion_incendios", "modelado")
DIR_DATOS = os.path.join(DATOS_BASE_DIR, "datos")
DIR_TABLAS = os.path.join(BASE_DIR, "resultados/tablas")
DIR_FIGURAS = os.path.join(BASE_DIR, "resultados/figuras")
os.makedirs(DIR_TABLAS, exist_ok=True)
os.makedirs(DIR_FIGURAS, exist_ok=True)

TARGET = "es_falsa_alarma"
SEMILLA = 42

# Para que el benchmark no tarde una eternidad, usa una fraccion representativa
# (ajusta si quieres el dataset completo, pero recuerda: esto se corre 2x por modelo)
FRACCION_BENCHMARK = 0.3


## 1. Monitor de recursos (RAM + GPU)

In [ ]:
class MonitorRecursos:
    def __init__(self, intervalo=0.5):
        self.intervalo = intervalo
        self._corriendo = False
        self._hilo = None
        self.muestras_ram_mb = []
        self.muestras_gpu_mem_mb = []
        self.muestras_gpu_util_pct = []

    def _leer_gpu(self):
        try:
            salida = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=memory.used,utilization.gpu",
                 "--format=csv,noheader,nounits"], timeout=2
            ).decode().strip().split("\n")[0].split(",")
            return float(salida[0].strip()), float(salida[1].strip())
        except Exception:
            return None, None

    def _loop(self):
        proceso = psutil.Process(os.getpid()) if HAY_PSUTIL else None
        while self._corriendo:
            if proceso is not None:
                self.muestras_ram_mb.append(proceso.memory_info().rss / 1e6)
            gpu_mem, gpu_util = self._leer_gpu()
            if gpu_mem is not None:
                self.muestras_gpu_mem_mb.append(gpu_mem)
                self.muestras_gpu_util_pct.append(gpu_util)
            threading.Event().wait(self.intervalo)

    def __enter__(self):
        self._corriendo = True
        self._hilo = threading.Thread(target=self._loop, daemon=True)
        self._hilo.start()
        return self

    def __exit__(self, *exc):
        self._corriendo = False
        self._hilo.join(timeout=2)

    def resumen(self):
        def _pico_prom(lista):
            return (max(lista), sum(lista) / len(lista)) if lista else (None, None)
        ram_pico, ram_prom = _pico_prom(self.muestras_ram_mb)
        gpu_mem_pico, gpu_mem_prom = _pico_prom(self.muestras_gpu_mem_mb)
        gpu_util_pico, gpu_util_prom = _pico_prom(self.muestras_gpu_util_pct)
        return {
            "ram_pico_mb": ram_pico, "gpu_mem_pico_mb": gpu_mem_pico,
            "gpu_util_promedio_pct": gpu_util_prom,
        }


## 2. Carga de datos

In [ ]:
train_pd = pd.read_parquet(os.path.join(DIR_DATOS, "train.parquet"))
test_pd  = pd.read_parquet(os.path.join(DIR_DATOS, "test.parquet"))

if FRACCION_BENCHMARK < 1.0:
    train_pd = train_pd.sample(frac=FRACCION_BENCHMARK, random_state=SEMILLA)
    test_pd = test_pd.sample(frac=FRACCION_BENCHMARK, random_state=SEMILLA)

X_train, y_train = train_pd.drop(columns=[TARGET]), train_pd[TARGET]
X_test,  y_test  = test_pd.drop(columns=[TARGET]),  test_pd[TARGET]
print(f"Benchmark con {len(y_train):,} filas de train, {len(y_test):,} de test")

resultados_benchmark = []


## 3. Random Forest: sklearn/CPU vs cuML/GPU

In [ ]:
from sklearn.ensemble import RandomForestClassifier as skRF
from sklearn.metrics import f1_score

# --- CPU (sklearn, forzado) ---
rf_cpu = skRF(n_estimators=300, max_depth=16, n_jobs=-1, class_weight="balanced", random_state=SEMILLA)
t0 = time.perf_counter()
with MonitorRecursos() as mon:
    rf_cpu.fit(X_train, y_train)
t_cpu = time.perf_counter() - t0
f1_cpu = f1_score(y_test, rf_cpu.predict(X_test))
r_cpu = mon.resumen()
print(f"RF CPU (sklearn): {t_cpu:.1f}s | F1={f1_cpu:.3f} | RAM pico={r_cpu['ram_pico_mb']:.0f}MB")

resultados_benchmark.append({
    "modelo": "Random Forest", "backend": "CPU (sklearn)", "tiempo_s": t_cpu, "f1": f1_cpu, **r_cpu
})

# --- GPU (cuML, si esta disponible) ---
try:
    import cudf
    from cuml.ensemble import RandomForestClassifier as cuRF

    Xtr_gpu = cudf.DataFrame.from_pandas(X_train.astype("float32"))
    ytr_gpu = cudf.Series(y_train.values.astype("int32"))
    Xte_gpu = cudf.DataFrame.from_pandas(X_test.astype("float32"))

    rf_gpu = cuRF(n_estimators=300, max_depth=16, n_streams=4, random_state=SEMILLA)
    t0 = time.perf_counter()
    with MonitorRecursos() as mon:
        rf_gpu.fit(Xtr_gpu, ytr_gpu)
    t_gpu = time.perf_counter() - t0
    pred_gpu = rf_gpu.predict(Xte_gpu)
    pred_gpu = pred_gpu.to_numpy() if hasattr(pred_gpu, "to_numpy") else pred_gpu
    f1_gpu = f1_score(y_test.values, pred_gpu)
    r_gpu = mon.resumen()
    print(f"RF GPU (cuML): {t_gpu:.1f}s | F1={f1_gpu:.3f} | speedup={t_cpu/t_gpu:.1f}x")

    resultados_benchmark.append({
        "modelo": "Random Forest", "backend": "GPU (cuML)", "tiempo_s": t_gpu, "f1": f1_gpu, **r_gpu
    })
except ImportError:
    print("cuML no disponible en este kernel -> solo se registro CPU para RF")


## 4. XGBoost: CPU vs GPU

In [ ]:
import xgboost as xgb

dtrain = xgb.DMatrix(X_train, label=y_train)
dtest  = xgb.DMatrix(X_test, label=y_test)
scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

params_base = {
    "objective": "binary:logistic", "eval_metric": "auc", "tree_method": "hist",
    "max_depth": 8, "eta": 0.1, "subsample": 0.8, "colsample_bytree": 0.8,
    "scale_pos_weight": scale_pos_weight, "seed": SEMILLA,
}

for device in ["cpu", "cuda"]:
    params = {**params_base, "device": device}
    t0 = time.perf_counter()
    try:
        with MonitorRecursos() as mon:
            modelo = xgb.train(params, dtrain, num_boost_round=300)
        t_xgb = time.perf_counter() - t0
        pred = (modelo.predict(dtest) >= 0.5).astype(int)
        f1_xgb = f1_score(y_test, pred)
        r_xgb = mon.resumen()
        etiqueta = "GPU (cuda)" if device == "cuda" else "CPU"
        print(f"XGBoost {etiqueta}: {t_xgb:.1f}s | F1={f1_xgb:.3f}")
        resultados_benchmark.append({
            "modelo": "XGBoost", "backend": etiqueta, "tiempo_s": t_xgb, "f1": f1_xgb, **r_xgb
        })
    except Exception as e:
        print(f"XGBoost {device} fallo: {e}")


## 5. Red Neuronal (MLP): CPU vs GPU

Mismo patron que RF/XGBoost: mismo modelo, mismos datos, forzando `cpu` vs `cuda`
explicitamente. Usa indexado vectorizado de tensores en vez de `DataLoader`
(evita el cuello de botella de iterar fila por fila que ya nos costo caro en el
pipeline principal — ver notas del proyecto).


In [ ]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

TARGET_COLS = [c for c in X_train.columns]
escalador_bench = StandardScaler()
X_train_esc = escalador_bench.fit_transform(X_train).astype(np.float32)
X_test_esc = escalador_bench.transform(X_test).astype(np.float32)
y_train_arr = y_train.values
y_test_arr = y_test.values

EPOCHS_BENCH = 2  # suficiente para medir tiempo por epoca de forma estable, sin tardar una eternidad en CPU
BATCH_SIZE_BENCH = 8192

def iterar_batches(X, y, batch_size, shuffle=False):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.long)
    n = X_t.shape[0]
    idx = torch.randperm(n) if shuffle else torch.arange(n)
    for i in range(0, n, batch_size):
        sel = idx[i:i + batch_size]
        yield X_t[sel], y_t[sel]

class MLPBenchmark(nn.Module):
    def __init__(self, n_entradas, n_clases=2):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(n_entradas, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, n_clases),
        )
    def forward(self, x):
        return self.red(x)

def entrenar_nn(device_str):
    device = torch.device(device_str)
    modelo = MLPBenchmark(n_entradas=X_train_esc.shape[1]).to(device)
    criterio = nn.CrossEntropyLoss()
    optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)

    t0 = time.perf_counter()
    with MonitorRecursos() as mon:
        for _ in range(EPOCHS_BENCH):
            modelo.train()
            for x, y in iterar_batches(X_train_esc, y_train_arr, BATCH_SIZE_BENCH, shuffle=True):
                x, y = x.to(device), y.to(device)
                optimizador.zero_grad()
                perdida = criterio(modelo(x), y)
                perdida.backward()
                optimizador.step()
    tiempo = (time.perf_counter() - t0) / EPOCHS_BENCH  # tiempo promedio POR EPOCA

    modelo.eval()
    preds = []
    with torch.no_grad():
        for x, y in iterar_batches(X_test_esc, y_test_arr, BATCH_SIZE_BENCH, shuffle=False):
            x = x.to(device)
            preds.extend(modelo(x).argmax(1).cpu().numpy())
    f1_nn = f1_score(y_test_arr, np.array(preds))
    return tiempo, f1_nn, mon.resumen()

for device_str, etiqueta in [("cpu", "CPU"), ("cuda", "GPU (cuda)")]:
    if device_str == "cuda" and not torch.cuda.is_available():
        print("GPU no disponible en este kernel, se omite ese caso")
        continue
    tiempo, f1_nn, r_nn = entrenar_nn(device_str)
    print(f"Red Neuronal {etiqueta}: {tiempo:.1f}s/epoca | F1={f1_nn:.3f}")
    resultados_benchmark.append({
        "modelo": "Red Neuronal", "backend": etiqueta, "tiempo_s": tiempo, "f1": f1_nn, **r_nn
    })


## 6. Tabla y gráfico de speedup

In [ ]:
tabla_bench = pd.DataFrame(resultados_benchmark)
tabla_bench.to_csv(os.path.join(DIR_TABLAS, "05_benchmark_cpu_vs_gpu.csv"), index=False)
tabla_bench


In [ ]:
# Calcular speedup por modelo (tiempo_cpu / tiempo_gpu)
filas_speedup = []
for modelo in tabla_bench["modelo"].unique():
    sub = tabla_bench[tabla_bench["modelo"] == modelo]
    cpu_row = sub[sub["backend"].str.contains("CPU")]
    gpu_row = sub[sub["backend"].str.contains("GPU")]
    if len(cpu_row) and len(gpu_row):
        t_cpu_m = cpu_row["tiempo_s"].values[0]
        t_gpu_m = gpu_row["tiempo_s"].values[0]
        filas_speedup.append({"modelo": modelo, "speedup": t_cpu_m / t_gpu_m,
                               "tiempo_cpu_s": t_cpu_m, "tiempo_gpu_s": t_gpu_m})

tabla_speedup = pd.DataFrame(filas_speedup)
tabla_speedup.to_csv(os.path.join(DIR_TABLAS, "06_speedup_cpu_vs_gpu.csv"), index=False)
print(tabla_speedup)

fig = px.bar(tabla_speedup, x="modelo", y="speedup", title="Speedup GPU vs CPU por modelo",
             text_auto=".1f")
fig.add_hline(y=1, line_dash="dash", annotation_text="sin ganancia (1x)")
fig.show()

plt.figure(figsize=(6,4))
plt.bar(tabla_speedup["modelo"], tabla_speedup["speedup"], color="#27ae60")
plt.axhline(1, linestyle="--", color="gray")
for i, v in enumerate(tabla_speedup["speedup"]):
    plt.text(i, v + 0.1, f"{v:.1f}x", ha="center")
plt.ylabel("Speedup (CPU / GPU)")
plt.title("Speedup GPU vs CPU por modelo")
plt.tight_layout()
plt.savefig(os.path.join(DIR_FIGURAS, "08_speedup_cpu_vs_gpu.png"), dpi=150)
plt.show()


## 7. Resumen para Robson

Este notebook entrega directamente lo que pidió:

| Lo que pidió | Dónde está |
|---|---|
| Tiempos CPU vs GPU (tabular) | `resultados/tablas/05_benchmark_cpu_vs_gpu.csv` |
| Uso de RAM/GPU durante entrenamiento | Columnas `ram_pico_mb`, `gpu_mem_pico_mb`, `gpu_util_promedio_pct` en la misma tabla |
| Precision/Recall/F1/matriz de confusión | F1 aquí; el resto ya está en `03_comparacion_modelos.csv` y las `*_matriz_confusion.png` del pipeline principal |
| Speedup / escalabilidad del entrenamiento | `resultados/tablas/06_speedup_cpu_vs_gpu.csv` y `figuras/08_speedup_cpu_vs_gpu.png` |

**Nota sobre la red neuronal:** el benchmark usa solo 2 épocas (en vez de las 8 del
entrenamiento final) para medir el tiempo por época de forma estable sin que la
corrida en CPU tarde una eternidad — el `tiempo_s` reportado ya es "por época",
directamente comparable entre CPU y GPU.
